# Workshop: Machine Learning for Aquatic Remote Sensing
**Event:** International Science Council (ISC) SCOR Workshop on Satellite Remote Sensing <br>
**Location:** Department of Marine Sciences, Berhampur University, India <br>
**Date:** Dec 2025 <br>
**Instructor:** Chintan B. Maniyar, PhD Candidate, University of Georgia (chintanmaniyar@uga.edu) <br>

---
**Note on Usage:**
This notebook was developed specifically for educational purposes within the SCOR workshop curriculum. The code and workflows demonstrate the application of Machine Learning to aquatic remote sensing data. While compliant with scientific best practices, users should rigorously validate these models before applying them to operational or published research.

### Import Packages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append(r'./../')
import utils # this is my custom toolkit for visualization

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler

### Define a Neural Network Class

In [ ]:
class SimpleRegressionNNDropoutBN2L(nn.Module):
    def __init__(self, input_size,  dropout_rate=0.3):
        super(SimpleRegressionNNDropoutBN2L, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),   # First hidden layer
            nn.BatchNorm1d(32),          # batch normalization
            nn.ReLU(),                   # activation
            nn.Dropout(p=dropout_rate),  # Dropout

            nn.Linear(32, 16),           # Second hidden layer
            nn.BatchNorm1d(16),          # batch normalization
            nn.ReLU(),                   # activation
            nn.Dropout(p=dropout_rate),  # Dropout

            nn.Linear(16, 1)             # Output layer (scalar output)
        )

    def forward(self, x):
        return self.model(x)

### Start with $R_{rs}$ data

In [ ]:
df_rrs = pd.read_csv('./../data/rrs_chla_olci.csv', index_col=0)
df_rrs

In [ ]:
X = df_rrs.drop('Chla', axis=1).to_numpy() # these are all the bands
y = df_rrs['Chla'].to_numpy().reshape(-1,1) # this is chl-a concentration
X.shape, y.shape # number of samples in X and y (sanity check)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
sns.histplot(y, kde=True, ax=ax[0], legend=False)
sns.histplot(y, kde=True, ax=ax[1], log_scale=True,legend=False)
ax[0].set_xlabel('Chl-a [$mg/m^3$]')
ax[1].set_xlabel('Chl-a [$mg/m^3$]')
ax[0].set_title(rf'$N={len(y)}$' + '\nLinear Scale')
ax[1].set_title(rf'$N={len(y)}$' + '\nLog Scale')
fig.suptitle('Chla Distribution (y)')

#### Data Pre-processing

Now, let us split the data first, and then we will perform standardization and scaling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777) # different students  try different value for random_state
X_train.shape, y_train.shape, X_test.shape, y_test.shape # notice the number of samples in each partition now. What do you observe?

Now, we shall perform normalization for our target variable Chl-*a* (y), and we shall perform robust scaling of all feature variables - bands (X). For the first run, you can skip these steps, to observe the impact of scaling on training neural networks. Skip this step, and train the neural network using raw, and unscaled data. Then come back here, uncomment this part of code, and re-train the network, and notice if you see any improvement. 

So for X, or the feature set: <br>
$X_{new} = \frac{X - median}{IQR}$

And for y, or the target variable: <br>
$y_{new} = log(y+1)$

In [ ]:
# first scale train data  ## uncomment this after first run
# X_train = RobustScaler().fit_transform(X_train)
# y_train = np.log1p(y_train)

In [ ]:
# now scale test data, and preserve the scalar for X to use  during inference time  ## uncomment this after first run
# scalar_x = RobustScaler()
# X_test = scalar_x.fit_transform(X_test)

Finally, we will cast our data to "tensors" - this is a special type of data type, that allows for faster matrix operations like dot and cross products on your CPU, which our deep learning package, PyTorch, uses.

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_train_tensor.shape, y_train_tensor.shape

We will further split our training data into training and validation, to update our weights every epoch.

In [ ]:
# Split Training data further into Train vs Validation (e.g., 80/20 split)
X_trn, X_val, y_trn, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.2, random_state=42)

#### Training the neural network

We will start with specifying the parameters of our neural network, such as the input size, loss function and the number of epochs.

In [ ]:
input_dim = X_train.shape[1]
model = SimpleRegressionNNDropoutBN2L(input_size=input_dim, dropout_rate=0.3)

In [ ]:
criterion = nn.SmoothL1Loss()  # Mean Squared Error
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # optimizer
num_epochs = 100

Before we start training, we will also convert our tensors into data loaders, train on batches of data points together (parallelly), instead of serially (one-by-one) on a single data point.

In [ ]:
# Create DataLoaders (Crucial for Batch Normalization to work well)
batch_size = 32
train_dataset = TensorDataset(X_trn, y_trn)
val_dataset   = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

We will now begin our training loop. If you recall from our last session, this is an iterative process, which the model does for however many times we have specified. Here, we have set our epochs to 100. So, our neural network will keep going back and forth 100 times, and each time, it will try to minimize the loss function, which we have chosen as Mean Square Error.

You can experiment by changing the  ```num_epochs``` parameter, as well as other parameters to see what happens to the learning, and how it progresses as each epoch progresses.

In [ ]:
train_losses = []  # List to store training loss per epoch
val_losses = []    # List to store validation loss per epoch

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()  # ENABLE Dropout and BatchNorm updates
    running_train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(X_batch)
        
        # Calculate loss
        loss = criterion(predictions, y_batch)
        
        # Backward pass & Optimizer step
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * X_batch.size(0)

    # Calculate average losses for this epoch
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # --- Validation Phase ---
    model.eval()  # DISABLE Dropout and BatchNorm stats update
    running_val_loss = 0.0
    
    with torch.no_grad(): # No gradient needed for validation
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            running_val_loss += loss.item() * X_batch.size(0)
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Train MSE: {epoch_train_loss:.4f} | Val MSE: {epoch_val_loss:.4f}")

To visualize the training process, we will plot a curve that shows how the training loss and validation loss varied with each epoch of training. If it got worse, then the model did not learn. If it got better, then the model learned with each epoch, how to predict Chl-*a*.

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_losses, color='green', label='Train Loss')
ax.plot(val_losses, color='red', label='Val Loss')
ax.legend()
ax.set_xlabel("Epochs")
ax.set_ylabel("Mean Squared Error (MSE)")

Now, let us evaluate our neural network using the same testing split, as we have been using for our linear and ML models. 

In [ ]:
model.eval() # Important: set to eval mode for predictions
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_pred_tensor = model(X_test_tensor)

# convert predictions back to numpy from tensor for easier use with plots
y_pred = y_pred_tensor.numpy()
# y_pred = np.expm1(y_pred) # do this only if you're normalizing the target variable

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

> *Question*: Did the neural network do as well as you thought it would? Why, or why not? Recall our last session about the *data-hungry* nature of neural networks.

> ## *Question*: Can you extend the code to use BR/LH instead of $R_{rs}$ for the same neural network? (Hint: Follow the section above)

In [ ]:
# read your data here

In [ ]:
# separate X and y here

In [ ]:
# plot chla distributions here

In [ ]:
# do train test split here

In [ ]:
# scale train here

In [ ]:
# scale test here

In [ ]:
# train val split here

In [ ]:
# input dims and model here

In [ ]:
# model parameters here

In [ ]:
# data loaders and batches here

In [ ]:
# training loop here

In [ ]:
# loss-epoch curve here

In [ ]:
# get predictions in model eval mode here

In [ ]:
# plot validation here